To follow along the treatment in the book, first run

```bash
uv run old_import_products.py
```

The main() function still performs a drop_all() followed by a create_all() as a quick way to reset the database, but keep in mind that recreating the entire database from scratch to make changes gets less viable the more tables there are in the database. Soon you will learn about how to improve this using database migration scripts.


## Many-to-Many Relationship Queries

As you can surely guess, the products and countries relationship objects added to the Country and Product models respectively also simplify the use of the many-to-many relationship in queries.

To try some queries, begin by starting a Python session and importing the needed components:

In [1]:
from db import Session
from models import Product, Manufacturer, Country

In [2]:
from sqlalchemy import select, func
session = Session()

Here is how to get a product and its countries:

In [3]:
p = session.scalar(
    select(Product)
    .where(Product.name == "Timex Sinclair 1000")
)

p

Product(138, "Timex Sinclair 1000", "Manufacturer(70, "Timex Sinclair")", 1982, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80")

Here the countries relationship uses the default lazy loader, so it implicitly runs a query to get the list of countries when the attribute is accessed for the first time.

Similarly, a country can report its products:

In [4]:
c = session.scalar(
    select(Country)
    .where(Country.name == 'Portugal')
)

c

Country(22, "Portugal")

In [5]:
c.products

[Product(138, "Timex Sinclair 1000", "Manufacturer(70, "Timex Sinclair")", 1982, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"),
 Product(139, "Timex Sinclair 1500", "Manufacturer(70, "Timex Sinclair")", 1982, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"),
 Product(140, "Timex Sinclair 2048", "Manufacturer(70, "Timex Sinclair")", 1984, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"),
 Product(141, "Timex Computer 2048", "Manufacturer(70, "Timex Sinclair")", 1984, [Country(22, "Portugal")], "Z80"),
 Product(142, "Timex Computer 2068", "Manufacturer(70, "Timex Sinclair")", 1983, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"),
 Product(143, "Komputer 2086", "Manufacturer(70, "Timex Sinclair")", 1986, [Country(22, "Portugal"), Country(23, "Poland")], "Z80")]

Moving on to something more complex, here is a query that returns all the products that have multiple countries, along with how many countries each has:

In [6]:
country_count = func.count(Country.id).label(None)
q = (select(Product, country_count)
     .join(Product.countries)
     .group_by(Product)
     .having(country_count >= 2)
     .order_by(Product.name)
     )

session.execute(q).all()

[(Product(143, "Komputer 2086", "Manufacturer(70, "Timex Sinclair")", 1986, [Country(22, "Portugal"), Country(23, "Poland")], "Z80"), 2),
 (Product(142, "Timex Computer 2068", "Manufacturer(70, "Timex Sinclair")", 1983, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"), 3),
 (Product(138, "Timex Sinclair 1000", "Manufacturer(70, "Timex Sinclair")", 1982, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"), 3),
 (Product(139, "Timex Sinclair 1500", "Manufacturer(70, "Timex Sinclair")", 1982, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"), 3),
 (Product(140, "Timex Sinclair 2048", "Manufacturer(70, "Timex Sinclair")", 1984, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"), 3)]

In [7]:
q = (select(Product, func.count(Country.id))
     .join(Product.countries)
     .group_by(Product)
     .having(func.count(Country.id) >= 2)
     .order_by(Product.name)
     )

session.execute(q).all()

[(Product(143, "Komputer 2086", "Manufacturer(70, "Timex Sinclair")", 1986, [Country(22, "Portugal"), Country(23, "Poland")], "Z80"), 2),
 (Product(142, "Timex Computer 2068", "Manufacturer(70, "Timex Sinclair")", 1983, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"), 3),
 (Product(138, "Timex Sinclair 1000", "Manufacturer(70, "Timex Sinclair")", 1982, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"), 3),
 (Product(139, "Timex Sinclair 1500", "Manufacturer(70, "Timex Sinclair")", 1982, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"), 3),
 (Product(140, "Timex Sinclair 2048", "Manufacturer(70, "Timex Sinclair")", 1984, [Country(1, "UK"), Country(3, "USA"), Country(22, "Portugal")], "Z80"), 3)]